# 🤖 Практикум 4. Разбиение выборки, baseline и первые метрики качества

После чтения и разведочного анализа данных необходимо перейти к постановке модели. Однако до сложных алгоритмов важно построить базовую линию сравнения и понять, как именно оценивается качество прогноза.


## 🧭 Введение и описание практикума

В этом практикуме рассматривается учебный набор данных по комбинированной парогазовой электростанции. Содержательный акцент сделан не на сложности алгоритма, а на методически корректной последовательности действий: выделение целевой переменной, разбиение выборки, построение baseline и интерпретация метрик качества.


In [ ]:
from pathlib import Path
import sys

import pandas as pd

# Добавляем корень репозитория в путь импорта, чтобы ноутбук запускался
# одинаково и из корня проекта, и из папки notebooks.
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from src.project_paths import project_root, sample_data_dir
from src.display_utils import display_callout, display_frame, display_metric_table, display_stage_note


import matplotlib.pyplot as plt
import numpy as np
from math import sqrt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from src.plot_utils import COURSE_COLORS, apply_course_plot_theme, format_axis

# Читаем данные и выделяем признаки и целевую переменную.
regression_path = sample_data_dir() / "regression" / "combined_cycle_power_plant.csv"
plant = pd.read_csv(regression_path)
X = plant[["AT", "V", "AP", "RH"]]
y = plant["PE"]

display_frame(
    pd.DataFrame(
        {
            "Роль": ["Целевая переменная", "Число признаков", "Размер набора данных"],
            "Значение": ["PE", X.shape[1], len(plant)],
        }
    )
)

# Разбиваем данные на обучающую и тестовую части.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Строим baseline-прогноз как среднее по обучающей выборке.
baseline_pred = np.full_like(y_test, fill_value=y_train.mean(), dtype=float)

# Обучаем линейную регрессию как первую содержательную модель.
model = LinearRegression()
model.fit(X_train, y_train)
model_pred = model.predict(X_test)

baseline_metrics = {
    "MAE": mean_absolute_error(y_test, baseline_pred),
    "RMSE": sqrt(mean_squared_error(y_test, baseline_pred)),
    "R²": r2_score(y_test, baseline_pred),
}
model_metrics = {
    "MAE": mean_absolute_error(y_test, model_pred),
    "RMSE": sqrt(mean_squared_error(y_test, model_pred)),
    "R²": r2_score(y_test, model_pred),
}

comparison = pd.DataFrame(
    [
        {"Модель": "Baseline по среднему", **baseline_metrics},
        {"Модель": "Линейная регрессия", **model_metrics},
    ]
)
display_stage_note(
    "Сравнение baseline и модели",
    "Здесь становится видно, дает ли обучаемая модель реальное преимущество по сравнению с простейшим ориентиром.",
)
display_frame(comparison)

# Строим диаграмму "факт - прогноз", чтобы увидеть геометрию ошибки.
apply_course_plot_theme()
fig, ax = plt.subplots(figsize=(6.5, 6))
ax.scatter(y_test, model_pred, color=COURSE_COLORS["accent"], alpha=0.55)
bounds = [min(y_test.min(), model_pred.min()), max(y_test.max(), model_pred.max())]
ax.plot(bounds, bounds, color=COURSE_COLORS["highlight"], linestyle="--", linewidth=1.5)
format_axis(
    ax,
    title="Сопоставление фактических и прогнозных значений",
    xlabel="Фактическая мощность, МВт",
    ylabel="Прогноз модели, МВт",
    subtitle="Пунктирная линия соответствует идеальному совпадению прогноза и факта",
)
plt.tight_layout()
plt.show()

display_metric_table(model_metrics)


### 📏 Как связать метрики и график с аналитическим выводом

Если линейная модель уменьшает `MAE` и `RMSE` относительно baseline, а облако точек на диаграмме тяготеет к диагонали, значит признаки действительно объясняют существенную часть вариации мощности. В противном случае следует либо пересматривать набор признаков, либо пробовать другую модельную схему.


## 📌 Практическое значение результата

Этот практикум показывает, что оценка модели начинается не с выбора «самого умного» алгоритма, а с правильно выстроенной процедуры сравнения. Когда baseline, метрики и графическая интерпретация собраны в одну схему, дальнейшее усложнение модели становится осмысленным.
